# Data Cleansing: All techniques and methods needed for modern and robust AI/ML

Welcome to the foundation of all successful Machine Learning. The adage "Garbage In, Garbage Out" (GIGO) is the absolute law of Data Science. No matter how advanced your neural network or ensemble model is, if it is fed noisy, incomplete, or unscaled data, it will fail to learn the underlying patterns.

Data Cleansing (or preprocessing) is the systematic process of identifying and resolving corrupt, inaccurate, or irrelevant data. In this tutorial, we will analytically break down the essential pipeline:
1. Handling Missing Data
2. Outlier Detection and Treatment
3. Categorical Encoding
4. Feature Scaling

Let's build a robust, production-ready pipeline using Python's core data ecosystem.

In [ ]:
# Cell 2: Imports and Environment Setup
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder

# Set pandas display options for better analytical visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## Step 1: Generating a Messy "Real-World" Dataset

To demonstrate data cleansing, we need dirty data. Real-world datasets suffer from missing values (NaNs), extreme outliers, and categorical variables represented as text. 

We will synthesize a dataset representing house prices to serve as our analytical sandbox.

In [ ]:
# Cell 4: Synthesizing Messy Data
np.random.seed(42)

data = {
    'Square_Feet': [1500, 2000, np.nan, 2500, 1200, 3000, 1800, 10000, 2200, 1900], # Notice the outlier: 10000
    'Bedrooms': [3, 4, 3, 5, np.nan, 4, 3, 8, 4, 3],
    'Neighborhood': ['Subrub', 'City', 'City', 'Rural', 'Suburb', np.nan, 'City', 'Rural', 'City', 'Suburb'], # Notice the typo 'Subrub'
    'Price': [300000, 450000, 350000, 500000, 250000, 600000, 380000, 2000000, 420000, 400000]
}

df = pd.DataFrame(data)
print("--- Raw Messy Dataset ---")
print(df)
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

## Step 2: Handling Missing Data (Imputation)

Machine learning algorithms generally cannot process `NaN` or `null` values. We have two primary choices:
1. **Deletion:** Drop rows/columns with missing data. (Only recommended if missing data is less than 5% and is Missing Completely At Random).
2. **Imputation:** Estimate the missing value based on the rest of the data.

For continuous variables, we typically impute using the Mean or Median. If the data is skewed by outliers, the Median is mathematically safer. For categorical variables, we impute using the Mode (most frequent value).

In [ ]:
# Cell 6: Implementing Imputation
df_clean = df.copy()

# 1. Correcting the typo first so our mode calculation is accurate
df_clean['Neighborhood'] = df_clean['Neighborhood'].replace('Subrub', 'Suburb')

# 2. Imputing Numerical Columns (Square_Feet and Bedrooms) using the Median
num_imputer = SimpleImputer(strategy='median')
df_clean[['Square_Feet', 'Bedrooms']] = num_imputer.fit_transform(df_clean[['Square_Feet', 'Bedrooms']])

# 3. Imputing Categorical Columns (Neighborhood) using the Mode
cat_imputer = SimpleImputer(strategy='most_frequent')
# SimpleImputer returns a 2D array, so we unravel it with .ravel()
df_clean['Neighborhood'] = cat_imputer.fit_transform(df_clean[['Neighborhood']]).ravel()

print("--- Data After Imputation ---")
print(df_clean)

## Step 3: Outlier Detection and Treatment

Outliers are extreme values that deviate significantly from the rest of the data. They can pull the mean away from the center and drastically alter the slope of regression lines.

A robust mathematical method for identifying outliers is the **Interquartile Range (IQR)** method. 
The IQR is the distance between the 75th percentile ($Q_3$) and the 25th percentile ($Q_1$):

$$IQR = Q_3 - Q_1$$

We define the boundaries for acceptable data as:
* Lower Bound = $Q_1 - 1.5 \times IQR$
* Upper Bound = $Q_3 + 1.5 \times IQR$

Points outside these bounds are capped (Winsorized) or removed.

In [ ]:
# Cell 8: Implementing IQR Outlier Handling
def clip_outliers_iqr(dataframe, column):
    """Clips outliers in a continuous column to the IQR bounds."""
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Clip limits the values to the specified min and max
    dataframe[column] = dataframe[column].clip(lower=lower_bound, upper=upper_bound)
    return dataframe

# Apply to our Square_Feet column (which has the massive 10,000 sq ft outlier)
df_clean = clip_outliers_iqr(df_clean, 'Square_Feet')

print("--- Data After Outlier Clipping ---")
print(df_clean[['Square_Feet', 'Price']])

## Step 4: Categorical Encoding

Algorithms require numerical matrices. They do not understand strings like "City" or "Suburb". We must map text to numbers.

**One-Hot Encoding** is the standard for nominal data (categories with no inherent order). It creates a new binary column (0 or 1) for every unique category. This prevents the model from assuming that category "2" is mathematically greater than category "1".

In [ ]:
# Cell 10: Implementing One-Hot Encoding
# pandas get_dummies is the fastest way to do this for data analysis
# drop_first=True prevents the "Dummy Variable Trap" (perfect multicollinearity)

df_encoded = pd.get_dummies(df_clean, columns=['Neighborhood'], drop_first=True)

# Convert boolean columns to integers (1/0) for broader compatibility with ML libraries
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print("--- Data After One-Hot Encoding ---")
print(df_encoded)

## Step 5: Feature Scaling

Features often have drastically different scales (e.g., Bedrooms range from 1-10, while Price ranges from 100,000-1,000,000). Distance-based algorithms (like KNN or SVM) and gradient descent optimization will be heavily biased towards features with larger magnitudes.

**Standardization (Z-score scaling)** transforms data to have a mean of 0 and a standard deviation of 1. It is highly effective for algorithms assuming a Gaussian distribution:

$$Z = \frac{x - \mu}{\sigma}$$

Where $x$ is the value, $\mu$ is the mean, and $\sigma$ is the standard deviation.

In [ ]:
# Cell 12: Implementing Feature Scaling
# We scale the features, but we typically DO NOT scale the target variable ('Price') 
# unless specifically required by the algorithm (like Neural Networks).

features_to_scale = ['Square_Feet', 'Bedrooms']

scaler = StandardScaler()
df_encoded[features_to_scale] = scaler.fit_transform(df_encoded[features_to_scale])

print("--- Final Cleansed and Scaled Dataset ---")
print(df_encoded)

## Conclusion

Your data is now mathematically stable. Through this analytical pipeline, we have:
1. Rescued lost information via **Median/Mode Imputation**.
2. Protected our future models from extreme variance using **IQR Clipping**.
3. Translated human text into machine-readable tensors via **One-Hot Encoding**.
4. Leveled the mathematical playing field using **Standardization**.

This `df_encoded` DataFrame is now ready to be split into train/test sets and fed directly into a scikit-learn predictive model.